# Naive Bayes — Exercises

Companion to the note [Naive Bayes](Naive%20Bayes.md).

Practise the conditional-independence assumption, MLE by counting, Laplace smoothing, log-space computation and the Gaussian, multinomial and Bernoulli variants, plus calibration and the link to logistic regression, by hand and from scratch in NumPy.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×5

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import logsumexp as sp_logsumexp
from scipy.stats import norm
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import log_loss
from sklearn.datasets import load_iris
rng = np.random.default_rng(0)
Xir, yir = load_iris(return_X_y=True)
# A tiny text corpus (1 = spam, 0 = ham)
docs = ["win money now", "win a free prize now", "free money free money", "claim your free prize",
        "meeting tomorrow at noon", "project meeting notes", "lunch tomorrow", "notes from the project lunch"]
y_docs = np.array([1, 1, 1, 1, 0, 0, 0, 0])
vec = CountVectorizer().fit(docs)
C_docs = vec.transform(docs).toarray()          # (8, V) word counts
vocab = vec.get_feature_names_out()
new_docs = ["free lunch now", "project prize meeting"]
C_new = vec.transform(new_docs).toarray()

## Part A · Concepts

### Exercise 1 · What is naive about it?
*🧠 Concept · ★☆☆*

State the naive Bayes assumption in words and as a formula. Give a concrete example where it is clearly false, and explain why the classifier can
still pick the right class.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Classification only needs the $\arg\max$, not correct probabilities.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Features are conditionally independent given the class: $p(x\mid y)=\prod_jp(x_j\mid y)$. In text, 'San' and 'Francisco' are strongly dependent
even within a class. The decision only needs the *ranking* of $p(y)\prod_jp(x_j\mid y)$ across classes to be right: errors in the probabilities often push all
classes in the same direction, so the $\arg\max$ survives even though the probabilities are badly off (over-confident).

</details>

### Exercise 2 · Pick the variant
*🧠 Concept · ★☆☆*

Which variant from the note's table (Gaussian, multinomial, Bernoulli) fits each case? (a) Sensor readings (temperature, pressure).
(b) Spam filtering with word counts. (c) Short SMS texts represented by 'word present / absent'. (d) Pixel intensities in $[0,255]$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Match the feature type: real-valued, counts, or binary. For (d), both a continuous and a binarised view are possible.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Gaussian. (b) Multinomial. (c) Bernoulli: it also uses the *absence* of words as evidence, which helps for short documents.
(d) Gaussian per pixel works poorly (pixel histograms are far from normal); binarising (pen/no pen) + Bernoulli, or treating intensities as counts (multinomial), is a common choice.

</details>

### Exercise 3 · Double counting and calibration
*🧠 Concept · ★★☆*

The note says NB probabilities are *poorly calibrated*. Suppose a feature is copied 5 times. What happens to the log-odds that NB computes, and why does
this make it over-confident? Why does [[Logistic Regression]] not have this problem?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Write $\log\frac{p(1\mid x)}{p(0\mid x)}$ as the prior log-odds plus a sum of per-feature log-likelihood ratios.

</details>

<details>
<summary><b>✅ Solution</b></summary>

NB log-odds $=\log\frac{\pi_1}{\pi_0}+\sum_j\log\frac{p(x_j\mid1)}{p(x_j\mid0)}$. Five identical copies add the same term five times: the evidence is counted
5×, the log-odds grow 5× and the probabilities are pushed to 0/1. Logistic regression fits the weights *jointly*: copies share one total weight
(the likelihood only depends on their sum), so the predicted probability is unchanged. You will measure this in Exercise 16.

</details>

### Exercise 4 · Generative vs discriminative pair
*🧠 Concept · ★★☆*

Naive Bayes and logistic regression are called a *generative–discriminative pair*. Explain what that means and when you would expect
NB to beat logistic regression (hint: training-set size).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ng & Jordan (2002): NB approaches its (higher) asymptotic error with $O(\log d)$ samples, logistic regression needs $O(d)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With Gaussian (shared variance) or Bernoulli features, NB's posterior has exactly the logistic form $\sigma(w^\top x+b)$ (Exercise 10), so both
search the same model family. NB fixes $w$ by counting (MLE of the joint), LR optimises the conditional likelihood. NB has lower variance and converges
with fewer samples, so it often wins for **small $N$**; as $N$ grows, LR, which is not biased by the false independence assumption, reaches a lower error.

</details>

## Part B · By hand

### Exercise 5 · A spam posterior
*✍️ By hand · ★☆☆*

$P(\text{spam})=0.4$. Word probabilities: $P(\text{free}\mid\text{spam})=0.3$, $P(\text{free}\mid\text{ham})=0.02$,
$P(\text{meeting}\mid\text{spam})=0.01$, $P(\text{meeting}\mid\text{ham})=0.05$. An e-mail contains both words (ignore all others).
Compute $P(\text{spam}\mid x)$.

In [ ]:
p_spam = None

_s = 0.4*0.3*0.01; _h = 0.6*0.02*0.05
check('P(spam | x)', p_spam, _s/(_s+_h), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Unnormalised scores: $0.4\cdot0.3\cdot0.01$ and $0.6\cdot0.02\cdot0.05$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Spam: $0.4\cdot0.3\cdot0.01=0.0012$. Ham: $0.6\cdot0.02\cdot0.05=0.0006$. Normalise: $P(\text{spam}\mid x)=\frac{0.0012}{0.0018}=\tfrac23\approx0.667$.

```python
p_spam = 0.0012 / 0.0018
```

</details>

### Exercise 6 · Laplace smoothing
*✍️ By hand · ★☆☆*

Vocabulary $V=4$: (free, win, meeting, lunch). In the spam class the word counts are $(5,3,0,0)$, so $n_k=8$.
Compute the MLE word probabilities and the Laplace-smoothed ones $\frac{n_{jk}+\alpha}{n_k+\alpha V}$ with $\alpha=1$.
Why is the unsmoothed estimate dangerous?

In [ ]:
theta_mle = None     # 4 numbers
theta_laplace = None # 4 numbers

_n = np.array([5,3,0,0.])
check('MLE', theta_mle, _n/_n.sum())
check('Laplace', theta_laplace, (_n+1)/(_n.sum()+4))

<details>
<summary><b>💡 Hint</b></summary>

Smoothed denominator: $8+1\cdot4=12$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

MLE: $(0.625,\ 0.375,\ 0,\ 0)$. Laplace: $(6,4,1,1)/12=(0.5,\ 0.333,\ 0.083,\ 0.083)$.
With the MLE, a single spam e-mail containing 'meeting' gets $P(x\mid\text{spam})=0$, i.e. $\log=-\infty$, so one unseen word vetoes all other evidence.

```python
theta_mle = [0.625, 0.375, 0, 0]
theta_laplace = [0.5, 1/3, 1/12, 1/12]
```

</details>

### Exercise 7 · Gaussian NB in 1-D
*✍️ By hand · ★★☆*

Class 0: $\mathcal N(0,1^2)$, prior $0.7$. Class 1: $\mathcal N(3,2^2)$, prior $0.3$. Compute $P(y=1\mid x=1.5)$.

In [ ]:
p1_gauss = None

_a = 0.7*norm.pdf(1.5, 0, 1); _b = 0.3*norm.pdf(1.5, 3, 2)
check('P(y=1 | x=1.5)', p1_gauss, _b/(_a+_b), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\mathcal N(x\mid\mu,\sigma^2)=\frac1{\sqrt{2\pi}\sigma}e^{-(x-\mu)^2/2\sigma^2}$. Note $x=1.5$ is 1.5 standard deviations from class 0 but only 0.75 from class 1.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p(1.5\mid0)=\frac{1}{\sqrt{2\pi}}e^{-1.125}\approx0.1295$, $p(1.5\mid1)=\frac{1}{2\sqrt{2\pi}}e^{-0.28125}\approx0.1506$.
Joint: $0.7\cdot0.1295=0.0907$ and $0.3\cdot0.1506=0.0452$. $P(y=1\mid x)=0.0452/0.1358\approx0.333$.
The likelihood favours class 1, but the prior flips the decision.

```python
p1_gauss = 0.3 * norm.pdf(1.5, 3, 2) / (0.3 * norm.pdf(1.5, 3, 2) + 0.7 * norm.pdf(1.5, 0, 1))
```

</details>

### Exercise 8 · Why the assumption saves parameters
*✍️ By hand · ★☆☆*

With $d=30$ binary features and $K=2$ classes, how many free parameters does a model of the **full** class-conditional $p(x\mid y)$ need, compared with
Bernoulli naive Bayes? (Count the class prior too.)

In [ ]:
params_full = None
params_nb = None

check('full joint', params_full, 2*(2**30-1)+1)
check('naive Bayes', params_nb, 2*30+1)

<details>
<summary><b>💡 Hint</b></summary>

A distribution over $2^{30}$ outcomes has $2^{30}-1$ free numbers.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Full: $K(2^d-1)+(K-1)=2(2^{30}-1)+1=2\,147\,483\,647$, impossible to estimate. NB: $Kd+(K-1)=61$. The independence assumption turns an exponential
number of parameters into a linear one: a strong bias that massively reduces variance.

```python
params_full = 2 * (2 ** 30 - 1) + 1
params_nb = 61
```

</details>

### Exercise 9 · MLE by counting
*✍️ By hand · ★★☆*

For Bernoulli NB with data $\{(x^{(n)},y^{(n)})\}$, show that maximising the log-likelihood gives
$\hat\pi_k=N_k/N$ and $\hat\theta_{jk}=\frac{\#\{n: y^{(n)}=k,\ x^{(n)}_j=1\}}{N_k}$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The log-likelihood splits into a sum of independent terms; for $\pi$ use a Lagrange multiplier for $\sum_k\pi_k=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\ell=\sum_n\Big[\log\pi_{y_n}+\sum_j x_{nj}\log\theta_{jy_n}+(1-x_{nj})\log(1-\theta_{jy_n})\Big]$.
Priors: maximise $\sum_kN_k\log\pi_k+\lambda(1-\sum_k\pi_k)$: $N_k/\pi_k=\lambda\Rightarrow\pi_k=N_k/N$.
Each $\theta_{jk}$ appears only in $c_{jk}\log\theta+(N_k-c_{jk})\log(1-\theta)$ with $c_{jk}$ = count of ones; the derivative
$c_{jk}/\theta-(N_k-c_{jk})/(1-\theta)=0$ gives $\theta=c_{jk}/N_k$. Training really is just counting.
Laplace smoothing is the posterior mean/MAP under a Beta/Dirichlet prior ([[Bayesian Inference]]).

</details>

### Exercise 10 · Gaussian NB is a linear classifier (sometimes)
*✍️ By hand · ★★★*

Two classes, Gaussian NB with per-feature variances $\sigma_j^2$ **shared between the classes**. Show that $p(y=1\mid x)=\sigma(w^\top x+b)$ and give $w_j$.
What happens if the variances differ between classes?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compute $\log\frac{p(x\mid1)\pi_1}{p(x\mid0)\pi_0}$ feature by feature; the $x_j^2$ terms cancel when the variances are equal.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log\frac{\mathcal N(x_j\mid\mu_{j1},\sigma_j^2)}{\mathcal N(x_j\mid\mu_{j0},\sigma_j^2)}=\frac{(x_j-\mu_{j0})^2-(x_j-\mu_{j1})^2}{2\sigma_j^2}
=\frac{\mu_{j1}-\mu_{j0}}{\sigma_j^2}x_j+\frac{\mu_{j0}^2-\mu_{j1}^2}{2\sigma_j^2}$.
Summing over $j$ and adding $\log\frac{\pi_1}{\pi_0}$ gives a linear function $a=w^\top x+b$ with $w_j=\frac{\mu_{j1}-\mu_{j0}}{\sigma_j^2}$, and $p(1\mid x)=\sigma(a)$:
the same form as [[Logistic Regression]]. With class-specific variances an $x_j^2$ term remains and the boundary is quadratic (like QDA in [[Linear Models for Classification]]).

</details>

### Exercise 11 · Underflow and log space
*✍️ By hand · ★☆☆*

A document has 1000 words, each with probability $0.01$ under a class. What is $\prod_jp(x_j\mid y)$ in floating point, and what is its log?

In [ ]:
log_lik = None

check('log-likelihood', log_lik, 1000*np.log(0.01), tol=1e-3)
print('product in float64:', 0.01**1000)

<details>
<summary><b>💡 Hint</b></summary>

The smallest positive float64 is about $5\times10^{-324}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The product is $10^{-2000}$, which underflows to exactly `0.0`, so all classes would get probability 0. In log space: $1000\log0.01=-4605.17$, perfectly representable.
Compare classes with the log scores, and normalise with log-sum-exp (Exercise 15).

```python
log_lik = 1000 * np.log(0.01)
```

</details>

## Part C · Code from scratch

### Exercise 12 · Gaussian naive Bayes
*💻 Code · ★★☆*

Implement `gnb_fit(X, y)` (priors, per-class means and **population** variances, plus sklearn's stabiliser $\epsilon=10^{-9}\max_j\operatorname{Var}(x_j)$ added to all variances)
and `gnb_predict_proba(params, X)` (work in log space). Compare with `GaussianNB` on Iris.

In [ ]:
def gnb_fit(X, y):
    # TODO: return (priors [K], means [K,d], vars [K,d])
    return None

def gnb_predict_proba(params, X):
    # TODO
    return None

gnb_params = gnb_fit(Xir, yir)
P_gnb = None if gnb_params is None else gnb_predict_proba(gnb_params, Xir)

check('predict_proba matches GaussianNB', P_gnb, GaussianNB().fit(Xir, yir).predict_proba(Xir), tol=1e-8)

<details>
<summary><b>💡 Hint 1</b></summary>

Log joint: $\log\pi_k-\tfrac12\sum_j\big[\log(2\pi\sigma_{jk}^2)+(x_j-\mu_{jk})^2/\sigma_{jk}^2\big]$, shape (N, K) with broadcasting.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Normalise rows with `L - logsumexp(L, axis=1, keepdims=True)` and exponentiate.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Training accuracy is about 0.96. Broadcasting `X[:, None, :] - means[None]` gives an (N, K, d) array, summed over the last axis.

```python
def gnb_fit(X, y):
    classes = np.unique(y)
    eps = 1e-9 * X.var(0).max()
    pri = np.array([np.mean(y == k) for k in classes])
    mu = np.array([X[y == k].mean(0) for k in classes])
    var = np.array([X[y == k].var(0) for k in classes]) + eps
    return pri, mu, var

def gnb_predict_proba(params, X):
    pri, mu, var = params
    L = np.log(pri) - 0.5 * np.sum(np.log(2 * np.pi * var)[None] + (X[:, None, :] - mu[None]) ** 2 / var[None], axis=2)
    return np.exp(L - sp_logsumexp(L, axis=1, keepdims=True))

gnb_params = gnb_fit(Xir, yir)
P_gnb = gnb_predict_proba(gnb_params, Xir)
print('training accuracy', np.mean(P_gnb.argmax(1) == yir))
```

</details>

### Exercise 13 · Multinomial NB for text
*💻 Code · ★★☆*

Using the word-count matrix `C_docs` (setup), implement multinomial NB with Laplace smoothing:
`mnb_fit(C, y, alpha)` returns log priors and $\log\theta_{kj}=\log\frac{n_{jk}+\alpha}{n_k+\alpha V}$; `mnb_predict_proba(params, C)`.
Classify `new_docs` and compare with `MultinomialNB(alpha=1)`.

In [ ]:
print(vocab)
def mnb_fit(C, y, alpha=1.0):
    # TODO: return (log_prior [K], log_theta [K, V])
    return None

def mnb_predict_proba(params, C):
    # TODO
    return None

mnb_params = mnb_fit(C_docs, y_docs)
P_mnb = None if mnb_params is None else mnb_predict_proba(mnb_params, C_new)

check('predict_proba matches MultinomialNB', P_mnb, MultinomialNB(alpha=1.0).fit(C_docs, y_docs).predict_proba(C_new), tol=1e-8)

<details>
<summary><b>💡 Hint</b></summary>

Log-likelihood of a count vector (up to the multinomial coefficient, which cancels): $\sum_j c_j\log\theta_{kj}$, i.e. `C @ log_theta.T`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

'free lunch now' is classified as spam (two spam words vs one ham word), 'project prize meeting' as ham. The multinomial coefficient is the same for every class,
so it cancels in the normalisation.

```python
def mnb_fit(C, y, alpha=1.0):
    classes = np.unique(y)
    counts = np.array([C[y == k].sum(0) for k in classes]) + alpha
    log_theta = np.log(counts / counts.sum(1, keepdims=True))
    log_prior = np.log(np.array([np.mean(y == k) for k in classes]))
    return log_prior, log_theta

def mnb_predict_proba(params, C):
    log_prior, log_theta = params
    L = C @ log_theta.T + log_prior
    return np.exp(L - sp_logsumexp(L, axis=1, keepdims=True))

mnb_params = mnb_fit(C_docs, y_docs)
P_mnb = mnb_predict_proba(mnb_params, C_new)
print(dict(zip(new_docs, P_mnb[:, 1].round(3))))
```

</details>

### Exercise 14 · Bernoulli NB
*💻 Code · ★★☆*

Binarise the counts (`B = C_docs > 0`) and implement Bernoulli NB with smoothing $\theta_{kj}=\frac{N_{kj}+\alpha}{N_k+2\alpha}$ ($N_{kj}$ = number of class-$k$ documents containing word $j$).
The likelihood includes **absent** words: $\prod_j\theta_{kj}^{x_j}(1-\theta_{kj})^{1-x_j}$. Compare with `BernoulliNB(alpha=1)` on `new_docs`.

In [ ]:
def bnb_fit(B, y, alpha=1.0):
    # TODO: return (log_prior, theta [K, V])
    return None

def bnb_predict_proba(params, B):
    # TODO
    return None

bnb_params = bnb_fit((C_docs > 0).astype(float), y_docs)
P_bnb = None if bnb_params is None else bnb_predict_proba(bnb_params, (C_new > 0).astype(float))

check('predict_proba matches BernoulliNB', P_bnb, BernoulliNB(alpha=1.0).fit(C_docs > 0, y_docs).predict_proba(C_new > 0), tol=1e-8)

<details>
<summary><b>💡 Hint</b></summary>

`L = B @ np.log(theta).T + (1 - B) @ np.log(1 - theta).T + log_prior`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The $(1-x_j)\log(1-\theta_{kj})$ term is what distinguishes Bernoulli from multinomial NB: not seeing a typical spam word is evidence for ham.
Denominator $N_k+2\alpha$ because each word has two outcomes (present/absent).

```python
def bnb_fit(B, y, alpha=1.0):
    classes = np.unique(y)
    theta = np.array([(B[y == k].sum(0) + alpha) / ((y == k).sum() + 2 * alpha) for k in classes])
    log_prior = np.log(np.array([np.mean(y == k) for k in classes]))
    return log_prior, theta

def bnb_predict_proba(params, B):
    log_prior, theta = params
    L = B @ np.log(theta).T + (1 - B) @ np.log(1 - theta).T + log_prior
    return np.exp(L - sp_logsumexp(L, axis=1, keepdims=True))

bnb_params = bnb_fit((C_docs > 0).astype(float), y_docs)
P_bnb = bnb_predict_proba(bnb_params, (C_new > 0).astype(float))
print(P_bnb.round(3))
```

</details>

### Exercise 15 · Log-sum-exp
*💻 Code · ★☆☆*

Implement `logsumexp(a)` $=\log\sum_ie^{a_i}$ that works for `a = [-1000, -1001, -1002]` and `a = [1000, 1000]`, and use it to turn the
log-scores `[-4605.2, -4607.0]` into posterior probabilities.

In [ ]:
def logsumexp(a):
    a = np.asarray(a, dtype=float)
    # TODO
    return None

post = None  # 2 probabilities from log scores [-4605.2, -4607.0]

for _a in ([-1000, -1001, -1002], [1000, 1000]):
    check(f'logsumexp{_a}', logsumexp(_a), sp_logsumexp(_a))
check('posterior', post, np.exp(np.array([-4605.2, -4607.0]) - sp_logsumexp([-4605.2, -4607.0])))

<details>
<summary><b>💡 Hint</b></summary>

$\log\sum e^{a_i}=m+\log\sum e^{a_i-m}$ with $m=\max_ia_i$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Shifting by the maximum makes the largest exponent $e^0=1$: no overflow, and at least one term does not underflow.
Posterior: $\sigma(1.8)\approx(0.858,\ 0.142)$; only the *difference* of log scores matters.

```python
def logsumexp(a):
    a = np.asarray(a, dtype=float)
    m = a.max()
    return m + np.log(np.sum(np.exp(a - m)))

_ls = np.array([-4605.2, -4607.0])
post = np.exp(_ls - logsumexp(_ls))
print(post)
```

</details>

### Exercise 16 · Measuring over-confidence
*💻 Code · ★★★*

Generate a 1-D two-class problem ($x\mid y\sim\mathcal N(\pm1, 1.5^2)$, 2000 train / 2000 test). Fit `GaussianNB` and `LogisticRegression` on (a) the single feature
and (b) the feature copied 5 times. Store the four test log-losses in `ll` as a dict with keys `'nb1','nb5','lr1','lr5'`.

In [ ]:
ll = None

if ll is None:
    check('NB gets worse with copies, LR does not', None, True)
else:
    check('NB gets worse with copies', ll['nb5'] > ll['nb1'] + 0.05, True)
    check('LR is (almost) unaffected', abs(ll['lr5'] - ll['lr1']) < 0.01, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.repeat(x[:, None], 5, axis=1)` makes the copies. `log_loss(y, model.predict_proba(X))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

NB's log-loss jumps (≈ 0.51 → ≈ 1.1) because the evidence is counted five times and probabilities become extreme, while its accuracy barely changes (same $\arg\max$ direction).
Logistic regression stays at ≈ 0.51: the five copies share one total weight. This is the calibration problem from Exercise 3.

```python
r = np.random.default_rng(5)
def make(n):
    y = r.integers(0, 2, n)
    return r.normal(2 * y - 1, 1.5)[:, None], y
xa, ya = make(2000); xb, yb = make(2000)
ll = {}
for tag, f in [('1', lambda x: x), ('5', lambda x: np.repeat(x, 5, axis=1))]:
    ll['nb' + tag] = log_loss(yb, GaussianNB().fit(f(xa), ya).predict_proba(f(xb)))
    ll['lr' + tag] = log_loss(yb, LogisticRegression().fit(f(xa), ya).predict_proba(f(xb)))
print({k: round(v, 3) for k, v in ll.items()})
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Naive Bayes](Naive%20Bayes.md).*